In [18]:
%reset -f
%reload_ext autoreload
%autoreload 2

In [19]:
import pandas as pd
import pyarrow.parquet as pq
import gc
import sys
sys.path.append("/home/zhuchen/poc/scorecard") 


In [20]:
cust_name = 'CR'

In [21]:
file_name = f'/home/zhuchen/poc/01-mashang/data_or/DeltaV1_data_all1_{cust_name}.parquet'

In [22]:
# 查看元数据
parquet_file = pq.ParquetFile(file_name)
print("Schema:")
print(parquet_file.schema)
print("\nMetadata:")
print(parquet_file.metadata)

# 获取所有列名
column_names = parquet_file.schema.names
print("Available columns:", column_names)

Schema:
required group field_id=-1 schema {
  optional binary field_id=-1 ms_no (String);
  optional binary field_id=-1 mobile_sha256 (String);
  optional binary field_id=-1 hs_date (String);
  optional binary field_id=-1 prod_flag (String);
  optional binary field_id=-1 sample_type (String);
  optional double field_id=-1 target1_mi;
  optional double field_id=-1 target2_mi;
  optional binary field_id=-1 stage (String);
  optional float field_id=-1 TZ_0000_m1;
  optional float field_id=-1 TZ_0000_m12;
  optional float field_id=-1 TZ_0000_m15;
  optional float field_id=-1 TZ_0000_m18;
  optional float field_id=-1 TZ_0000_m2;
  optional float field_id=-1 TZ_0000_m24;
  optional float field_id=-1 TZ_0000_m3;
  optional float field_id=-1 TZ_0000_m4;
  optional float field_id=-1 TZ_0000_m5;
  optional float field_id=-1 TZ_0000_m6;
  optional float field_id=-1 TZ_0000_m9;
  optional float field_id=-1 TZ_0000_w1;
  optional float field_id=-1 TZ_0000_w2;
  optional float field_id=-1 TZ_0000_w3

In [23]:
fea_list = column_names[8:]

In [24]:
fea_list.pop(-1)

'__index_level_0__'

In [25]:
from new_tools import iv_report


result_list2 = []
result_list1 = []
length = 2000
for i in range(0,(len(fea_list) // length) + 1):
    start, end = i * length, min((i + 1) * length, len(fea_list))
    print(start, ' - ', end)
    temp_fea_list = fea_list[start:end]

    temp_data_all = pd.read_parquet(file_name, 
                    columns=['target1_mi','target2_mi']+temp_fea_list,
                    engine='pyarrow')  # 或 engine='fastparquet'
    temp_data_all['weight'] = 1
    temp_data_all['target'] = 'train'
    # temp_data_all.loc[(pd.to_datetime(temp_data_all['backDateTime']) >= pd.to_datetime('2024-06-01')) & (temp_data_all['target'] =='train'),'target'] = 'oot'
    iv_calculator1 = iv_report.IVCalculator(temp_data_all.fillna(-999),label='target1_mi',target='target',keep_list=temp_fea_list, max_leaf_nodes=6, min_samples_leaf=0.05)
    iv_calculator2 = iv_report.IVCalculator(temp_data_all.fillna(-999),label='target2_mi',target='target',keep_list=temp_fea_list, max_leaf_nodes=6, min_samples_leaf=0.05)
    del temp_data_all
    iv_df1 = iv_calculator1.iv_report(use_thread=True,max_workers=20)
    iv_df2 = iv_calculator2.iv_report(use_thread=True,max_workers=20)
    # iv_df['dif'] = iv_df['train_iv']/iv_df['oot_iv']
    result_list1.append(iv_df1)
    result_list2.append(iv_df2)
    del iv_calculator1,iv_df1,iv_calculator2,iv_df2
    gc.collect()

0  -  2000


初始化IV计算器,获取各个数据集的索引...
初始化完成
初始化IV计算器,获取各个数据集的索引...
初始化完成


多线程特征计算IV: 100%|██████████| 2000/2000 [00:19<00:00, 104.39it/s]


2000  -  4000
初始化IV计算器,获取各个数据集的索引...
初始化完成
初始化IV计算器,获取各个数据集的索引...
初始化完成


多线程特征计算IV: 100%|██████████| 2000/2000 [00:19<00:00, 102.81it/s]


...

多线程特征计算IV: 100%|██████████| 2000/2000 [00:19<00:00, 101.75it/s]


12000  -  14000
初始化IV计算器,获取各个数据集的索引...
初始化完成
初始化IV计算器,获取各个数据集的索引...
初始化完成


多线程特征计算IV: 100%|██████████| 2000/2000 [00:21<00:00, 92.86it/s] 


14000  -  15160
初始化IV计算器,获取各个数据集的索引...
初始化完成
初始化IV计算器,获取各个数据集的索引...
初始化完成


多线程特征计算IV: 100%|██████████| 1160/1160 [00:10<00:00, 107.39it/s]


In [26]:
target1_iv_df = pd.concat(result_list1)
target2_iv_df = pd.concat(result_list2)
target1_iv_df.rename(columns={'train_iv':'total_iv'},inplace=True)
target2_iv_df.rename(columns={'train_iv':'total_iv'},inplace=True)
target1_iv_df.sort_values(by='total_iv',ascending=False,inplace=True)
target2_iv_df.sort_values(by='total_iv',ascending=False,inplace=True)
target1_iv_df.to_csv(f'/home/zhuchen/poc/01-mashang/data_or/DeltaV1_{cust_name}_target1_iv.csv',index=False)
target2_iv_df.to_csv(f'/home/zhuchen/poc/01-mashang/data_or/DeltaV1_{cust_name}_target2_iv.csv',index=False)
